# Movie Success Prediction
### Predicting Commercial Performance to Support Studio Greenlight Decisions

**Final Project** | Machine Learning (Binary Classification)

---

## Table of Contents
1. [Load and Merge](#1.-Load-and-merge)
2. [Data Cleaning](#2.-Data-cleaning)
3. [Parsing Nested Fields](#3.-Parse-nested-fields)
4. [Target Definition](#4.-Target-definition)
5. [Exploratory Analysis](#5.-Exploratory-analysis)
6. [Leakage Controls](#6.-Leakage-controls)
7. [Time-Based Split](#7.-Time-based-train/test-split)
8. [Feature Engineering](#8.-Feature-engineering)
9. [Greenlight Models](#9.-Greenlight-models)
10. [Feature Importance](#10.-Feature-importance)
11. [Critic Reception Analysis](#11.-Critic-reception-analysis)
12. [Conclusions and Limitations](#12.-Conclusions-and-limitations)

---

## Business Problem
Studios commit large budgets before knowing whether a film will recover its cost. This project estimates the likelihood that a new release is commercially successful, so decision-makers can prioritize and de-risk their slate.

## Research Question
> Does external critic reception (Rotten Tomatoes, Metacritic) improve predictions of commercial success beyond what is known at greenlight stage?

## Approach
| Item | Design choice |
|---|---|
| Task | Binary classification |
| Target | `success = 1` if revenue is at least 2.5x budget (a $100M-revenue definition is computed for comparison) |
| Data | TMDB 5000 Movies and Credits (Kaggle), enriched via the TMDB API and OMDb API |
| Validation | Time-based split: train on older films, test on the most recent 20% |
| Metrics | PR-AUC (primary), F1, recall, precision, ROC-AUC, confusion matrix |
| Models | Baseline, logistic regression, gradient boosting |

**Two prediction moments**
1. **Greenlight model:** only information available before release (budget, genres, cast, director, studio, release timing).
2. **Post-screening model:** greenlight features plus critic scores.

**Required files in `data/`:** `tmdb_5000_movies.csv`, `tmdb_5000_credits.csv`, and optionally `enriched.csv` (produced by `enrich.py`).

In [ ]:
import ast, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, ConfusionMatrixDisplay)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold", "font.size": 10})
np.random.seed(0)
pd.set_option("display.max_columns", 50)

DATA = Path("data")
SUCCESS_ROI = 2.5          # success if revenue >= SUCCESS_ROI * budget
SUCCESS_REVENUE = 100e6    # alternative definition
MIN_BUDGET = 100_000       # budgets below this are almost certainly data errors
MIN_REVENUE = 100_000
TEST_FRACTION = 0.2        # most recent 20% of films become the test set
SEED = 42

## 1. Load and merge

In [ ]:
movies = pd.read_csv(DATA / "tmdb_5000_movies.csv")
credits = pd.read_csv(DATA / "tmdb_5000_credits.csv")

# credits uses movie_id; movies uses id. Drop duplicate title column before merging.
df = movies.merge(credits.drop(columns="title").rename(columns={"movie_id": "id"}),
                  on="id", how="inner")
print(movies.shape, credits.shape, "->", df.shape)
df.head(2)

## 2. Data cleaning

- Remove duplicate IDs and unreleased films.
- `budget` or `revenue` of 0 means **missing**, not zero. We drop those rows (never impute the target).
- Absurdly small values (e.g. a $1 budget) are treated as data errors.

In [ ]:
n0 = len(df)
df = df.drop_duplicates(subset="id")
df = df[df["status"] == "Released"].copy()
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])

for c in ["budget", "revenue", "runtime"]:
    df[c] = df[c].replace(0, np.nan)

print("budget missing:", df.budget.isna().mean().round(2), "| revenue missing:", df.revenue.isna().mean().round(2))
df = df[(df.budget >= MIN_BUDGET) & (df.revenue >= MIN_REVENUE)].copy()
print(f"{n0} -> {len(df)} rows after cleaning")

## 3. Parse nested fields

Genres, studios, keywords, cast and crew are stored as JSON strings. They are parsed into Python lists, keeping only the director and the top three billed actors.

In [ ]:
def parse(x):
    """Parse a JSON/Python-literal string into a list; return [] on failure."""
    if isinstance(x, list):
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            continue
    return []

def names(x, n=None):
    out = [d.get("name") for d in parse(x) if d.get("name")]
    return out[:n] if n else out

def director(x):
    for d in parse(x):
        if d.get("job") == "Director":
            return d.get("name")
    return None

df["genres_list"] = df["genres"].apply(names)
df["studios_list"] = df["production_companies"].apply(names)
df["keywords_list"] = df["keywords"].apply(lambda x: [k.lower() for k in names(x)])
df["director"] = df["crew"].apply(director)
df["cast_top3"] = df["cast"].apply(lambda x: [d["name"] for d in sorted(parse(x), key=lambda d: d.get("order", 99))[:3]])
df["lead_actor"] = df["cast_top3"].apply(lambda l: l[0] if l else None)

# Simple sequel / franchise flag from TMDB keywords or a numbered title
df["is_sequel"] = (df["keywords_list"].apply(lambda l: "sequel" in l)
                   | df["title"].str.contains(r"(?:\b[2-9]\b|\bII\b|\bIII\b|Part )", regex=True)).astype(int)
df[["title", "director", "lead_actor", "genres_list", "is_sequel"]].head()

## 4. Target definition

A flat $100M threshold ignores budget (a $200M film grossing $100M is a flop). We use an ROI-based target as the main definition.

In [ ]:
df["roi"] = df["revenue"] / df["budget"]
df["success"] = (df["roi"] >= SUCCESS_ROI).astype(int)
df["success_100m"] = (df["revenue"] >= SUCCESS_REVENUE).astype(int)

print("success (ROI >= 2.5x):  ", df.success.mean().round(3))
print("success (>= $100M):   ", df.success_100m.mean().round(3))
print("agreement between definitions:", (df.success == df.success_100m).mean().round(3))

## 5. Exploratory analysis

Descriptive checks of how success varies with budget, release timing and genre. (Descriptive only: no modelling decisions are fitted on these views.)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Budget quintiles
q = pd.qcut(df["budget"], 5, labels=["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (highest)"])
df.groupby(q)["success"].mean().plot.bar(ax=axes[0], color="#3b6ea5", rot=30)
axes[0].set_title("Success rate by budget quintile"); axes[0].set_ylabel("Success rate"); axes[0].set_xlabel("")

# Release month
df.groupby(df["release_date"].dt.month)["success"].mean().plot.bar(ax=axes[1], color="#3b6ea5", rot=0)
axes[1].set_title("Success rate by release month"); axes[1].set_xlabel("Month"); axes[1].set_ylabel("")

# Genre
gx = df.explode("genres_list").dropna(subset=["genres_list"]).groupby("genres_list")["success"].agg(["mean", "size"])
gx[gx["size"] >= 50]["mean"].sort_values().plot.barh(ax=axes[2], color="#3b6ea5")
axes[2].set_title("Success rate by genre (n >= 50)"); axes[2].set_ylabel("")

for ax in axes: ax.axhline(df["success"].mean(), color="crimson", ls="--", lw=1)
plt.suptitle("Dashed line = overall success rate", y=1.02, fontsize=10)
plt.tight_layout(); plt.show()

## 6. Leakage controls

These columns are measured **after** release and are driven by the film's success, so they are excluded from every model:

`popularity`, `vote_count`, `vote_average`, `revenue`, `roi`.

Director and lead-actor history features are computed from **earlier films only** (`shift()` before the expanding mean).

In [ ]:
LEAKY = ["popularity", "vote_count", "vote_average", "revenue", "roi", "success", "success_100m"]

df = df.sort_values("release_date").reset_index(drop=True)
df["log_roi"] = np.log(df["roi"]).clip(-3, 5)

def prior_stats(col):
    """Mean log-ROI and film count for the person's films released strictly before this one."""
    g = df.groupby(col)["log_roi"]
    mean = g.transform(lambda s: s.shift().expanding().mean())
    count = df.groupby(col).cumcount()
    return mean, count

df["director_prior_log_roi"], df["director_n_prior"] = prior_stats("director")
df["lead_prior_log_roi"], df["lead_n_prior"] = prior_stats("lead_actor")

## 7. Time-based train/test split

A chronological split mimics real use: the model is trained on past films and evaluated on newer ones.

In [ ]:
cutoff = df["release_date"].quantile(1 - TEST_FRACTION)
train = df[df.release_date < cutoff].copy()
test = df[df.release_date >= cutoff].copy()
print("cutoff:", cutoff.date(), "| train:", len(train), "| test:", len(test))
print("success rate  train:", train.success.mean().round(3), " test:", test.success.mean().round(3))

## 8. Feature engineering

Genre and studio vocabularies are taken **from the training set only**. Rare studios are dropped (they become all-zero = "other").

In [ ]:
GENRES = sorted({g for l in train.genres_list for g in l})
studio_counts = pd.Series([s for l in train.studios_list for s in l]).value_counts()
TOP_STUDIOS = studio_counts.head(20).index.tolist()

def make_X(d):
    X = pd.DataFrame(index=d.index)
    X["log_budget"] = np.log1p(d["budget"])
    X["runtime"] = d["runtime"]
    X["release_month"] = d["release_date"].dt.month
    X["is_summer"] = d["release_date"].dt.month.isin([5, 6, 7]).astype(int)
    X["is_holiday"] = d["release_date"].dt.month.isin([11, 12]).astype(int)
    X["is_english"] = (d["original_language"] == "en").astype(int)
    X["is_sequel"] = d["is_sequel"]
    X["n_genres"] = d["genres_list"].apply(len)
    X["n_studios"] = d["studios_list"].apply(len)
    for g in GENRES:
        X["genre_" + g] = d["genres_list"].apply(lambda l: int(g in l))
    for s in TOP_STUDIOS:
        X["studio_" + s] = d["studios_list"].apply(lambda l: int(s in l))
    for c in ["director_prior_log_roi", "director_n_prior", "lead_prior_log_roi", "lead_n_prior"]:
        X[c] = d[c]
    return X

X_train, X_test = make_X(train), make_X(test)
y_train, y_test = train["success"], test["success"]
assert not set(LEAKY) & set(X_train.columns)
print("features:", X_train.shape[1])

## 9. Greenlight models

Accuracy would mislead on imbalanced data, so we report **PR-AUC** (compare against the success rate, which is what a random model scores), **F1**, **recall** and a confusion matrix.

In [ ]:
def evaluate(name, model, Xtr, ytr, Xte, yte, show_cm=False):
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    res = {"model": name,
           "PR-AUC": average_precision_score(yte, proba),
           "ROC-AUC": roc_auc_score(yte, proba),
           "F1": f1_score(yte, pred),
           "precision": precision_score(yte, pred, zero_division=0),
           "recall": recall_score(yte, pred)}
    if show_cm:
        ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=["flop", "success"], cmap="Blues")
        plt.title(name); plt.show()
    return res

def models():
    return {
        "Baseline (prior)": DummyClassifier(strategy="prior"),
        "Logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                             LogisticRegression(max_iter=2000, class_weight="balanced")),
        "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", max_depth=3,
                                                            learning_rate=0.05, max_iter=200, random_state=SEED),
    }

results = [evaluate(n, m, X_train, y_train, X_test, y_test, show_cm=(n != "Baseline (prior)"))
           for n, m in models().items()]
res_greenlight = pd.DataFrame(results).set_index("model").round(3)
print("Random-model PR-AUC would be about", y_test.mean().round(3))
res_greenlight

## 10. Feature importance

Permutation importance on the held-out set (model-agnostic). If you install `shap`, `shap.TreeExplainer` is a nice next step.

In [ ]:
gb = models()["Gradient boosting"].fit(X_train, y_train)
imp = permutation_importance(gb, X_test, y_test, scoring="average_precision",
                             n_repeats=20, random_state=SEED)
imp = pd.Series(imp.importances_mean, index=X_test.columns).sort_values().tail(15)
imp.plot.barh(figsize=(7, 5), title="Permutation importance (PR-AUC drop)")
plt.tight_layout(); plt.show()

## 11. Critic reception analysis

Needs `data/enriched.csv` from `enrich.py`. For a fair comparison, both models are evaluated on the **same subset** of films that have critic scores.

In [ ]:
enriched_path = DATA / "enriched.csv"
if not enriched_path.exists():
    print("Run enrich.py first to create data/enriched.csv, then re-run this section.")
else:
    enr = pd.read_csv(enriched_path)[["id", "rt_critic", "metacritic", "us_certification"]]
    tr = train.merge(enr, on="id", how="left")
    te = test.merge(enr, on="id", how="left")
    tr.index, te.index = X_train.index, X_test.index   # keep row alignment with X frames

    print("critic score coverage  train:", tr.rt_critic.notna().mean().round(2),
          " test:", te.rt_critic.notna().mean().round(2))

    # Same subset for both models
    m_tr, m_te = tr.rt_critic.notna() | tr.metacritic.notna(), te.rt_critic.notna() | te.metacritic.notna()
    Xa_tr, Xa_te = X_train[m_tr], X_test[m_te]
    Xb_tr = Xa_tr.assign(rt_critic=tr.loc[m_tr, "rt_critic"], metacritic=tr.loc[m_tr, "metacritic"])
    Xb_te = Xa_te.assign(rt_critic=te.loc[m_te, "rt_critic"], metacritic=te.loc[m_te, "metacritic"])
    ya_tr, ya_te = y_train[m_tr], y_test[m_te]

    rows = []
    for n in ["Logistic regression", "Gradient boosting"]:
        rows.append({**evaluate(f"{n} | greenlight", models()[n], Xa_tr, ya_tr, Xa_te, ya_te)})
        rows.append({**evaluate(f"{n} | + critic scores", models()[n], Xb_tr, ya_tr, Xb_te, ya_te)})
    print("test films:", len(ya_te), "| base rate:", ya_te.mean().round(3))
    display(pd.DataFrame(rows).set_index("model").round(3))

## 12. Conclusions and limitations

### Summary of results
The table below consolidates the greenlight results. Compare the **PR-AUC** of each model with the baseline (the test-set success rate) to judge real predictive lift.

In [ ]:
summary = res_greenlight[["PR-AUC", "ROC-AUC", "F1", "recall", "precision"]].copy()
summary["PR-AUC lift vs. baseline"] = (summary["PR-AUC"] / y_test.mean()).round(2)
display(summary)
best = summary["PR-AUC"].idxmax()
print(f"Best greenlight model by PR-AUC: {best} ({summary.loc[best, 'PR-AUC']:.3f} vs. baseline {y_test.mean():.3f})")

### Key takeaways
*(Complete after running the notebook on the full dataset.)*
- **Predictive power:** how much better do the models perform than the baseline success rate?
- **Main drivers:** which features lead the permutation-importance chart (typically budget, release timing, franchise status, prior track record)?
- **Value of critic scores:** does adding Rotten Tomatoes / Metacritic improve PR-AUC and recall on the same set of films?

### Limitations
- **Sample size:** a few hundred test films make metrics noisy; differences between models may not be statistically meaningful.
- **Survivorship and selection:** films with missing budget or revenue were removed, which skews the sample toward better-documented releases.
- **Nominal currency:** budgets and revenues are not inflation-adjusted. The ROI-based target partly mitigates this, but absolute budget features may drift over time.
- **Revenue scope:** TMDB revenue reflects reported box office and excludes streaming, home video and merchandising.
- **Coverage bias:** critic scores are missing for some films, so the critic comparison uses a subset.
- **Association, not causation:** results describe predictive patterns and should not be read as causal effects.

### Recommended next steps
1. Repeat the analysis with the `success_100m` target and compare conclusions.
2. Extend to a three-class target (low / medium / high ROI).
3. Tune the decision threshold to the business cost of a missed hit versus a wasted greenlight.
4. Add keywords, MPAA certification and release-window competition as features.
5. Use rolling-origin (expanding-window) cross-validation for more stable estimates.
6. Add SHAP explanations for stakeholder-friendly model interpretation.